In [1]:
import torch
t = torch.tensor([[1., 2., 3.], [4., 5., 6.]])

print(t.shape, t.dtype)
print(t.ndim, t.numel())
print(torch.zeros(2, 3).shape)
print(torch.arange(6).reshape(2, 3))

torch.Size([2, 3]) torch.float32
2 6
torch.Size([2, 3])
tensor([[0, 1, 2],
        [3, 4, 5]])


In [4]:
# dim degani nechta dimention dan tashkil topgan degani
# dim = 0 ( bu ustunlar )
# dim = 1 ( bu qatorlar )
print(t.sum(dim=0).tolist())
print(t.sum(dim=1).tolist())
print(t.mean().item())
print((t * 2).tolist())
print((t @ t.T).tolist())

[5.0, 7.0, 9.0]
[6.0, 15.0]
3.5
[[2.0, 4.0, 6.0], [8.0, 10.0, 12.0]]
[[14.0, 32.0], [32.0, 77.0]]


In [6]:
print(t[0, 1].item())
print(t[:, 1].tolist())
print(t[t > 3].tolist())
print(t.reshape(3, 2).tolist())
print(t.reshape(-1).tolist())
print(t.reshape(3, -1).shape)

2.0
[2.0, 5.0]
[4.0, 5.0, 6.0]
[[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]
[1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
torch.Size([3, 2])


In [7]:
import numpy as np
print(torch.tensor([1, 2, 3]).dtype)
print((torch.tensor([1, 2, 3]) / 2).tolist())
print(torch.tensor([1, 2, 3]).float().mean().item())
a = np.array([1., 2., 3.])
t1 = torch.from_numpy(a)
t2 = torch.tensor(a)
a[0] = 100
print(t1.tolist(), t2.tolist())
print(t1.dtype)

torch.int64
[0.5, 1.0, 1.5]
2.0
[100.0, 2.0, 3.0] [1.0, 2.0, 3.0]
torch.float64


In [11]:
x = torch.tensor(2.)
w = torch.tensor(3., requires_grad=True)
b = torch.tensor(1., requires_grad=True)
y = w * x + b
loss = ( y - 10 ) ** 2
print(y.item(), loss.item())
loss.backward()
print(w.grad.item(), b.grad.item())
print(x.grad)

7.0 9.0
-12.0 -6.0
None


In [12]:
loss = (w * x + b - 10) ** 2
loss.backward()
print(w.grad.item(), b.grad.item())
w.grad.zero_()
b.grad.zero_()
print(w.grad.item(), b.grad.item())
# gradientlar yig'ilib boradi, buni oldini olishimiz uchun zeros_() qilishimiz kerak
# undan keyin backward ni amalga oshiramiz.

-24.0 -12.0
0.0 0.0


In [13]:
w = torch.tensor(3., requires_grad=True)
b = torch.tensor(1., requires_grad=True)
for step in range(3):
  loss = ( w * x + b - 10 ) ** 2
  loss.backward()
  with torch.no_grad():
    w-= 0.1 * w.grad
    b-= 0.1 * b.grad
  w.grad.zero_()
  b.grad.zero_()
  print(step, round(loss.item(), 4),
        round(w.item(), 4), round(b.item(), 4))

0 9.0 4.2 1.6
1 0.0 4.2 1.6
2 0.0 4.2 1.6


In [15]:
w = torch.tensor(3., requires_grad=True)
b = torch.tensor(1., requires_grad=True)
for step in range(3):
  loss = ( w * x + b - 10 ) ** 2
  loss.backward(retain_graph=True)
  loss.backward()
  with torch.no_grad():
    w-= 0.1 * w.grad
    b-= 0.1 * b.grad
  # w.grad.zero_()
  # b.grad.zero_()
  print(step, round(loss.item(), 4),
        round(w.item(), 4), round(b.item(), 4))

0 9.0 5.4 2.2
1 9.0 5.4 2.2
2 9.0 3.0 1.0


In [16]:
x = torch.tensor(3.)
w = torch.tensor(2., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
for step in range(3):
  loss = ( w * x + b - 4 ) ** 2
  loss.backward()
  with torch.no_grad():
    w-= 0.05 * w.grad
    b-= 0.05 * b.grad
  w.grad.zero_()
  b.grad.zero_()
  print(step, round(loss.item(), 4),
        round(w.item(), 4), round(b.item(), 4))

0 4.0 1.4 -0.2
1 0.0 1.4 -0.2
2 0.0 1.4 -0.2


In [17]:
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
lin = nn.Linear(13, 16)
print(tuple(lin.weight.shape), tuple(lin.bias.shape))
xb = torch.randn(5, 13)
print(tuple(lin(xb).shape))
print(torch.allclose(lin(xb), xb @ lin.weight.T + lin.bias))
print(sum(p.numel() for p in lin.parameters()))

(16, 13) (16,)
(5, 16)
True
224


In [20]:
class WineNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(13, 16)
    self.fc2 = nn.Linear(16, 3)
  def forward(self, x):
    return self.fc2(F.relu(self.fc1(x)))

torch.manual_seed(0)
model = WineNet()
print(tuple(model(xb).shape))
print(sum(p.numel() for p in model.parameters()))
for name, p in model.named_parameters():
  print(name, tuple(p.shape))

(5, 3)
275
fc1.weight (16, 13)
fc1.bias (16,)
fc2.weight (3, 16)
fc2.bias (3,)


In [22]:
torch.manual_seed(0)
seq = nn.Sequential(nn.Linear(13, 16), nn.ReLU(), nn.Linear(16, 3))
print(torch.allclose(model(xb), seq(xb)))
print(list(model.state_dict().keys()))
torch.save(model.state_dict(), "wine.pt")
torch.manual_seed(5); new = WineNet()
print(torch.allclose(model(xb), new(xb)))
new.load_state_dict(torch.load("wine.pt"))
print(torch.allclose(model(xb), new(xb)))

True
['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias']
False
True


In [23]:
torch.manual_seed(0)
lin = nn.Linear(4, 2)
print(tuple(lin.weight.shape))
print(sum(p.numel() for p in lin.parameters()))


(2, 4)
10


In [24]:
torch.manual_seed(0)
sec = nn.Sequential(
    nn.Linear(13, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 3)
)
print(sum(p.numel() for p in sec.parameters()))

1027


In [28]:
class WineNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(13, 8)
    self.fc2 = nn.Linear(8, 3)
  def forward(self, x):
    return self.fc2(F.relu(self.fc1(x)))

torch.manual_seed(0)
model = WineNet()
print(sum(p.numel() for p in model.parameters()))


139


In [31]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

# data load
X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
sc = StandardScaler().fit(X_train)

# use pytorch
A = torch.tensor(sc.transform(X_train), dtype=torch.float32)
B = torch.tensor(sc.transform(X_test), dtype=torch.float32)
ya, yb = torch.tensor(y_train), torch.tensor(y_test)

ds = TensorDataset(A, ya)
dl = DataLoader(ds, batch_size=32, shuffle=False)
print(A.shape, A.dtype, ya.dtype)
print([len(xb) for xb, _ in dl])

torch.Size([142, 13]) torch.float32 torch.int64
[32, 32, 32, 32, 14]


In [32]:
def train(lr=0.1, epochs=30, bs=32, seed=0, zero=True):
  torch.manual_seed(seed)
  m = WineNet()
  opt = torch.optim.SGD(m.parameters(), lr=lr)
  g = torch.Generator().manual_seed(seed)
  dl = DataLoader(ds, batch_size = bs, shuffle=True, generator=g)
  hist=[]
  for ep in range(epochs):
    m.train(); total = 0
    for xb, yb_ in dl:
      loss = F.cross_entropy(m(xb), yb_)
      if zero: opt.zero_grad()
      loss.backward(); opt.step()
      total += loss.item() * len(xb)
    hist.append(total / len(ds))
  return m, hist

In [33]:
def accuracy(m, X, Y):
  m.eval()
  with torch.no_grad():
    return (m(X).argmax(1) == Y).float().mean().item()
model, hist = train()
print([round(h, 3) for h in hist[::5]])
tr = accuracy(model, A, ya)
te = accuracy(model, B, yb)
print(round(tr, 4), round(te, 4))

[1.06, 0.554, 0.194, 0.102, 0.067, 0.05]
0.993 0.9722


In [34]:
for zero in (True, False):
  accs, ls = [], []
  for s in range(5):
    m, h = train(seed=s, zero=zero)
    accs.append(accuracy(m, B, yb))
    ls.append(h[-1])
  print(zero, round(sum(ls) / 5, 4),
        round(sum(accs) / 5, 4), round(min(accs), 4))

True 0.0372 0.9889 0.9722
False 0.0 0.9611 0.9444


In [35]:
# task 4
def train(lr=0.01, epochs=30, bs=32, seed=0, zero=True):
  torch.manual_seed(seed)
  m = WineNet()
  opt = torch.optim.SGD(m.parameters(), lr=lr)
  g = torch.Generator().manual_seed(seed)
  dl = DataLoader(ds, batch_size = bs, shuffle=True, generator=g)
  hist=[]
  for ep in range(epochs):
    m.train(); total = 0
    for xb, yb_ in dl:
      loss = F.cross_entropy(m(xb), yb_)
      if zero: opt.zero_grad()
      loss.backward(); opt.step()
      total += loss.item() * len(xb)
    hist.append(total / len(ds))
  return m, hist

def accuracy(m, X, Y):
  m.eval()
  with torch.no_grad():
    return (m(X).argmax(1) == Y).float().mean().item()
model, hist = train()
print([round(h, 3) for h in hist[::5]])
tr = accuracy(model, A, ya)
te = accuracy(model, B, yb)
print(round(tr, 4), round(te, 4))

[1.083, 1.042, 0.999, 0.954, 0.906, 0.853]
0.8451 0.8056


In [36]:
# task 4
def train(lr=0.01, epochs=100, bs=32, seed=0, zero=True):
  torch.manual_seed(seed)
  m = WineNet()
  opt = torch.optim.SGD(m.parameters(), lr=lr)
  g = torch.Generator().manual_seed(seed)
  dl = DataLoader(ds, batch_size = bs, shuffle=True, generator=g)
  hist=[]
  for ep in range(epochs):
    m.train(); total = 0
    for xb, yb_ in dl:
      loss = F.cross_entropy(m(xb), yb_)
      if zero: opt.zero_grad()
      loss.backward(); opt.step()
      total += loss.item() * len(xb)
    hist.append(total / len(ds))
  return m, hist

def accuracy(m, X, Y):
  m.eval()
  with torch.no_grad():
    return (m(X).argmax(1) == Y).float().mean().item()
model, hist = train()
print([round(h, 3) for h in hist[::5]])
tr = accuracy(model, A, ya)
te = accuracy(model, B, yb)
print(round(tr, 4), round(te, 4))

[1.083, 1.042, 0.999, 0.954, 0.906, 0.853, 0.8, 0.745, 0.69, 0.633, 0.577, 0.521, 0.468, 0.418, 0.372, 0.332, 0.297, 0.267, 0.242, 0.22]
0.9718 0.9722


In [37]:
# task 4
def train(lr=0.01, epochs=100, bs=124, seed=5, zero=True):
  torch.manual_seed(seed)
  m = WineNet()
  opt = torch.optim.SGD(m.parameters(), lr=lr)
  g = torch.Generator().manual_seed(seed)
  dl = DataLoader(ds, batch_size = bs, shuffle=True, generator=g)
  hist=[]
  for ep in range(epochs):
    m.train(); total = 0
    for xb, yb_ in dl:
      loss = F.cross_entropy(m(xb), yb_)
      if zero: opt.zero_grad()
      loss.backward(); opt.step()
      total += loss.item() * len(xb)
    hist.append(total / len(ds))
  return m, hist

def accuracy(m, X, Y):
  m.eval()
  with torch.no_grad():
    return (m(X).argmax(1) == Y).float().mean().item()
model, hist = train()
print([round(h, 3) for h in hist[::5]])
tr = accuracy(model, A, ya)
te = accuracy(model, B, yb)
print(round(tr, 4), round(te, 4))

[1.114, 1.079, 1.046, 1.014, 0.982, 0.952, 0.923, 0.895, 0.868, 0.842, 0.816, 0.791, 0.766, 0.741, 0.717, 0.694, 0.671, 0.648, 0.625, 0.602]
0.8169 0.8056
